# Protection & Audit (Intervalle)

The core SDC workflow, end to end:

1. **Compute** tables from microdata
2. **Suppress** unsafe cells (OPT / MOD / RND)
3. **Audit** — compute the *realized feasibility intervals* for each
   suppressed cell (the legacy `intervalle.exe`, now in-process)
4. **Export** the protected table (CSV or legacy INTERMEDIATE)

## The audit in one paragraph

A suppressed cell has a range of values it could take while staying
consistent with the published totals — its **realized lower/upper
bounds** (the *feasibility interval*). Protection requires sufficient
uncertainty to meet the configured protection requirements; a narrow
interval can reveal the value. In legacy Tau-Argus audit ran in a **separate**
`intervalle.exe`; the rewrite ports it into the engine (`TauAuditJj`
in the `csp` submodule), so it runs **in-process** over a temporary
`.JJ` file with nothing external to install.

## Setup

In [1]:
from pathlib import Path
import pytauargus

# Locate the repo's data/ fixtures (works from notebooks/ or repo root)
_here = Path.cwd()
DATA = next(
    (p / "data") for p in [_here, *_here.parents]
    if (p / "data" / "tau_testW.asc").exists()
)
OUT = _here / "out"          # scratch dir for notebook outputs (gitignored)
OUT.mkdir(exist_ok=True)
print(f"pytauargus {pytauargus.__version__}")
print(f"data : {DATA}")
print(f"out  : {OUT}")

pytauargus 0.2.0
data : /Users/trygve/local-code/tauargus/data
out  : /Users/trygve/local-code/tauargus/bindings/python/out


## 1. Compute the tables

Run the batch. The NK (n-k) safety rule flags cells whose value is
dominated by k other cells — these start life as `unsafe_rule`.

In [2]:
from pytauargus.engine import run_batch
from pytauargus.batch import Suppress
from collections import Counter

STATUS_NAMES = {
    1: "safe", 2: "safe_manual", 3: "unsafe_rule", 4: "unsafe_peep",
    5: "unsafe_freq", 6: "unsafe_zero", 7: "unsafe_singleton",
    8: "unsafe_singleton_manual", 9: "unsafe_manual", 10: "protect_manual",
    11: "secondary_unsafe", 12: "secondary_unsafe_manual",
    13: "empty_nonstructural", 14: "empty",
}

eng = run_batch(DATA / "TestRecode.arb")
tau = eng._tau
n = tau.get_total_table_size(0)[0]
before = Counter(tau.get_table_cell_status(0, j) for j in range(n))
print(f"Table 1 — {n} cells, pre-suppression:")
for code_, cnt in sorted(before.items()):
    print(f"  {STATUS_NAMES.get(code_, code_):<24} {cnt}")

Table 1 — 126 cells, pre-suppression:
  safe                     104
  unsafe_rule              1
  empty                    21


## 2. Suppress

Suppression methods available in headless mode:

| Method | Call | Notes |
|--------|------|-------|
| **OPT** | `eng.suppress(Suppress(kind="OPT", tab_no=N))` | Optimal, LP-based. `opt_max_time=0` = no limit (default). |
| **MOD** | `eng.suppress(Suppress(kind="MOD", tab_no=N))` | Modular, heuristic. `mod_max_time=0` (default). |
| **RND** | `eng.suppress(Suppress(kind="RND", tab_no=N, rnd_base=B))` | Controlled rounding; `B` must be ≥ 10 000 000. |

> `opt_max_time`/`mod_max_time` are in **minutes**; `0` means no limit.
> Positive limits are covered by repeated-run regression tests.
> These examples use the default `0` so they can finish without a deadline.
>
> `GH` (global hiding), `NET` (network) and `CTA` need external
> executables and are **not** available headless.

In [3]:
# Suppress table 1 (tab_no is 1-based in the API, as in the .arb grammar)
eng.suppress(Suppress(kind="OPT", tab_no=1))

after = Counter(tau.get_table_cell_status(0, j) for j in range(n))
print("Table 1 — after OPT suppression:")
for code_, cnt in sorted(after.items()):
    print(f"  {STATUS_NAMES.get(code_, code_):<24} {cnt}")

Table 1 — after OPT suppression:
  safe                     101
  unsafe_rule              1
  secondary_unsafe         3
  empty                    21


### RND (controlled rounding) — alternate flow

Controlled rounding needs a base ≥ 10 000 000. The engine computes a
suitable default from the table's cell sizes:

In [4]:
# Suppression mutates state — use a fresh engine for the alternate flow
eng2 = run_batch(DATA / "TestRecode.arb")
base = eng2._min_round_base(0)
print(f"rounding base: {base}")
eng2.suppress(Suppress(kind="RND", tab_no=1, rnd_base=base))
print("RND applied OK")

rounding base: 10000000
RND applied OK


## 3. Audit (Intervalle)

`eng.audit(tab)` (tab is 0-based) computes the **realized
feasibility interval** for every suppressed cell of `tab`. For each
it solves two LPs — minimize the cell, then maximize it, subject to
the published totals and cell bounds — and stores the bounds on the
cell (readable via `get_table_cell` as the last two entries).

Returns a list of dicts, one per suppressed cell:

| Key | Meaning |
|-----|---------|
| `cell` | Cell index (row-major) |
| `min` / `max` | Realized lower / upper bound |
| `value` | Original cell value, withheld from the release table |
| `status` | `u` = primary, `m` = secondary |
| `unsafe` | `True` if the ported audit reports insufficient protection |

In [5]:
rows = eng.audit(0)
print(f"{len(rows)} suppressed cells audited")
print()
for r in rows:
    flag = "   ⚠ UNDER-PROTECTED" if r["unsafe"] else ""
    print(f"  cell {r['cell']:3d}  interval=[{r['min']:>10.1f}, {r['max']:>10.1f}]  "
          f"value={r['value']:>10.1f}  {r['status']}{flag}")

4 suppressed cells audited

  cell  78  interval=[      -0.0,    30118.0]  value=   18150.0  m
  cell  79  interval=[   69831.0,    99949.0]  value=   81799.0  m
  cell  96  interval=[      -0.0,    30118.0]  value=   11968.0  u
  cell  97  interval=[  126854.0,   156972.0]  value=  145004.0  m


### Reading the result

`unsafe=False` means the cell passes the ported audit's protection-level
check. The feasibility interval normally contains the original value;
protection comes from sufficient uncertainty, not from excluding it.

`unsafe=True` means the audit reports insufficient protection. Review
the suppression pattern and protection requirements before publishing.

### Try it yourself

- Suppress table 2 instead: `eng.suppress(Suppress(kind="OPT", tab_no=2))`
  then `eng.audit(1)` — a 12 726-cell table.
- Compare OPT vs MOD on table 1 and diff the status counts.

## 4. Export

Two public export paths:

| Format | Call | Notes |
|--------|------|-------|
| CSV (published values) | `eng.write_table(WriteTable(tab_no=1, output_type=1, file=…))` | `output_type=1` is CSV |
| **INTERMEDIATE** (legacy audit file) | `eng.write_intermediate_table(tab, path, with_audit=True)` | One row per cell + realized bounds |

The INTERMEDIATE (WRITETABLE type-5) file is the legacy audit
format: values, statuses, and — with `with_audit=True` (the legacy
`AR+` option) — the realized feasibility bounds and relative-width
percentages for every cell.

In [6]:
from pytauargus.batch import WriteTable

csv_path = str(OUT / "table1.csv")
eng.write_table(WriteTable(tab_no=1, output_type=1, file=csv_path))
print(f"wrote {csv_path} ({(OUT / 'table1.csv').stat().st_size} bytes)")
print()
print("\n".join((OUT / "table1.csv").read_text().splitlines()[:5]))

wrote /Users/trygve/local-code/tauargus/bindings/python/out/table1.csv (700 bytes)

,"Total","Nr"," 1"," 2"," 3","Os"," 4"," 5"," 6"," 7","Ws"," 8"," 9","10","Zd","11","12","99"
"Total",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-
" 1",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-
" 6",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-
" 7",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-


In [7]:
tab_path = OUT / "table1.tab"
eng.write_intermediate_table(0, str(tab_path), with_audit=True)
lines = tab_path.read_text().splitlines()
print(f"wrote {tab_path} ({tab_path.stat().st_size} bytes, {len(lines) - 1} rows)")
print()
print(f"header: {lines[0][:100]}…")
print(f"row 1:  {lines[1][:100]}…")

wrote /Users/trygve/local-code/tauargus/bindings/python/out/table1.tab (10976 bytes, 125 rows)

header: "Total";"Total";16847646.84;42723;16847646.84;10.66;175677.00;141482.00;S;0.01;0.01;0.00;0.00;0.00;0…
row 1:  "Total";"Nr";4373664.00;11395;4373664.00;9.34;43587.00;43293.00;S;0.01;0.01;0.00;0.00;0.00;0;0;0…


## Gotchas & known limitations

- **OPT/MOD time limits:** `opt_max_time` / `mod_max_time` are in
  minutes; `0` means no limit. Repeated fresh-engine runs with a
  one-minute limit pass; deadline expiry is not covered by that test.
- **GH / NET / CTA** require external executables and are not
  available in headless mode.
- **`audit()` mutates state** (it stores the realized bounds on
  the table); run it after your final suppression pass.
- The CLI mirrors every step above: `tauargus run`, `tables`,
  `suppress`, `round`, `audit`, `save`.

## Summary

You've now:

1. Computed tables and inspected the safety-rule results
2. Applied OPT suppression (and RND as an alternate flow)
3. Audited suppressed cells — realized feasibility intervals
4. Exported the protected table (CSV + INTERMEDIATE)

**Next:** [03_generators.ipynb](03_generators.ipynb) — generate
`.arb` / `.rda` / `.hrc` input files from Python.